# In2STEM Project 1: Finding Trends in Galaxy Populations

In this notebook, the star formation rate and main-sequence classification steps are already completed in code.

Your goal is to investigate questions like:

- What do galaxies below the main sequence have in common?
- Are above-main-sequence galaxies dustier, bluer, or stronger in H-alpha emission?
- Does redshift, stellar mass, colour, or emission-line strength change between classes?

This is an exploration notebook. Try lots of plots, make masks, compare groups, and write down what each plot suggests.

## 1. Load the Data

This notebook uses a richer teaching sample made from the original DESI low-redshift table. It contains the same 10,000 galaxies as the other Binder notebooks, but with extra columns for trends work.

Some useful columns are:

- `Z`: redshift;
- `LOGMSTAR`: log stellar mass;
- `SFR_CALCULATED`: star formation rate from H-alpha;
- `AV`: dust attenuation from the stellar population fit;
- `VDISP`: velocity dispersion;
- `U_R_COLOUR`, `G_R_COLOUR`, `R_Z_COLOUR`: galaxy colours from absolute magnitudes;
- `HALPHA_EW`: H-alpha equivalent width;
- `HALPHA_SN`, `HBETA_SN`, `OIII_5007_SN`, `NII_6584_SN`: signal-to-noise values;
- `LOG_OIII_HBETA`, `LOG_NII_HALPHA`, `LOG_OII_HALPHA`: simple emission-line ratios.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set_theme(style='whitegrid', context='notebook')

In [ ]:
data = pd.read_csv('data/low_z_trends_sample.csv')

data.head()

In [ ]:
# See all available columns.
list(data.columns)

## 2. Everything Up To Main-Sequence Classification

The next cells calculate the log star formation rate, fit a simple main-sequence relation, and classify every galaxy.

We will use the same classification idea as before:

- `above main sequence`: more than 0.3 dex above the fitted line;
- `main sequence`: within 0.3 dex of the fitted line;
- `below main sequence`: more than 0.3 dex below the fitted line.

In this notebook, galaxies below the fitted main sequence are useful **quiescent candidates**. Be careful with the wording: this is a teaching sample and a simple classification, not a perfect scientific quiescent-galaxy catalogue.

In [ ]:
# Make log(SFR), safely avoiding log10 of zero or negative numbers.
sfr = np.asarray(data['SFR_CALCULATED'])
data['LOG_SFR'] = np.where(sfr > 0, np.log10(sfr), np.nan)

log_mass = np.asarray(data['LOGMSTAR'])
log_sfr = np.asarray(data['LOG_SFR'])

valid_main_sequence = np.isfinite(log_mass) & np.isfinite(log_sfr)
print('Number of galaxies used for the fit:', valid_main_sequence.sum())

In [ ]:
# Fit a simple straight-line main sequence.
fit_gradient, fit_intercept = np.polyfit(
    log_mass[valid_main_sequence],
    log_sfr[valid_main_sequence],
    deg=1
)

print(f'log(SFR) = {fit_gradient:.3f} log(M*) + {fit_intercept:.3f}')

# Calculate expected log(SFR), expected SFR, and offset from the fitted main sequence.
data['MS_LOG_SFR_EXPECTED'] = fit_gradient * data['LOGMSTAR'] + fit_intercept
data['MS_SFR_EXPECTED'] = 10 ** data['MS_LOG_SFR_EXPECTED']
data['MS_OFFSET'] = data['LOG_SFR'] - data['MS_LOG_SFR_EXPECTED']

# Classify galaxies.
data['MS_CLASS'] = 'main sequence'
data.loc[data['MS_OFFSET'] > 0.3, 'MS_CLASS'] = 'above main sequence'
data.loc[data['MS_OFFSET'] < -0.3, 'MS_CLASS'] = 'below main sequence'
data.loc[~valid_main_sequence, 'MS_CLASS'] = 'not classified'

# Shorter labels are useful for plot legends.
data['MS_CLASS_SHORT'] = data['MS_CLASS'].replace({
    'above main sequence': 'above',
    'main sequence': 'main',
    'below main sequence': 'below',
    'not classified': 'unclassified'
})

data['MS_CLASS'].value_counts()

## 3. Main Sequence Plots

Start by checking the classification visually. These plots are your map of the galaxy population.

In [ ]:
class_colours = {
    'below main sequence': 'royalblue',
    'main sequence': 'darkgreen',
    'above main sequence': 'crimson',
    'not classified': 'lightgrey'
}

mass_grid = np.linspace(data['LOGMSTAR'].min(), data['LOGMSTAR'].max(), 200)
fit_line = fit_gradient * mass_grid + fit_intercept

plt.figure(figsize=(8, 6))
for label, colour in class_colours.items():
    mask = data['MS_CLASS'] == label
    if mask.sum() == 0:
        continue
    plt.scatter(data.loc[mask, 'LOGMSTAR'], data.loc[mask, 'LOG_SFR'],
                s=10, alpha=0.45, color=colour, label=f'{label} ({mask.sum()})')

plt.plot(mass_grid, fit_line, color='black', linewidth=2.5, label='Fitted main sequence')
plt.plot(mass_grid, fit_line + 0.3, color='black', linestyle='--', linewidth=1)
plt.plot(mass_grid, fit_line - 0.3, color='black', linestyle='--', linewidth=1)
plt.xlabel(r'log($M_* / M_\odot$)')
plt.ylabel(r'log(SFR / $M_\odot$ yr$^{-1}$)')
plt.title('Main-Sequence Classification')
plt.legend()
plt.show()

In [ ]:
# A 2D histogram is useful when a scatter plot has too many points.
plt.figure(figsize=(8, 6))
plt.hist2d(
    data.loc[valid_main_sequence, 'LOGMSTAR'],
    data.loc[valid_main_sequence, 'LOG_SFR'],
    bins=55,
    cmap='magma',
    cmin=1
)
plt.plot(mass_grid, fit_line, color='cyan', linewidth=2.5, label='Fitted main sequence')
plt.plot(mass_grid, fit_line + 0.3, color='cyan', linestyle='--', linewidth=1)
plt.plot(mass_grid, fit_line - 0.3, color='cyan', linestyle='--', linewidth=1)
plt.colorbar(label='Number of galaxies')
plt.xlabel(r'log($M_* / M_\odot$)')
plt.ylabel(r'log(SFR / $M_\odot$ yr$^{-1}$)')
plt.title('Main Sequence as a 2D Histogram')
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(7, 4))
sns.countplot(data=data, x='MS_CLASS_SHORT', order=['below', 'main', 'above'], palette=['royalblue', 'darkgreen', 'crimson'], hue='MS_CLASS_SHORT', legend=False)
plt.xlabel('Main-sequence class')
plt.ylabel('Number of galaxies')
plt.title('How Many Galaxies Are in Each Class?')
plt.show()

## 4. Quick Summary Table

Before making lots of plots, compare the median properties of each class. This can reveal useful leads.

In [ ]:
properties_to_compare = [
    'Z', 'LOGMSTAR', 'LOG_SFR', 'SFR_CALCULATED', 'MS_OFFSET',
    'AV', 'VDISP', 'U_R_COLOUR', 'G_R_COLOUR', 'R_Z_COLOUR',
    'HALPHA_EW', 'HALPHA_SN', 'HBETA_SN', 'LOG_NII_HALPHA', 'LOG_OIII_HBETA'
]

summary = data.groupby('MS_CLASS')[properties_to_compare].median(numeric_only=True)
summary.round(3)

In [ ]:
# Difference from main-sequence galaxies.
main_values = summary.loc['main sequence']
(summary - main_values).round(3)

## 5. Helper Functions for Easy Masks and Plots

A **mask** is a True/False filter. You can use masks to select galaxies with certain properties.

Examples:

```python
massive = data['LOGMSTAR'] >= 10.5
nearby = data['Z'] <= 0.1
red = data['G_R_COLOUR'] >= 0.7
quiescent_candidates = data['MS_CLASS'] == 'below main sequence'
```

For pandas/NumPy masks:

- use `&` for **and**;
- use `|` for **or**;
- put each condition inside brackets.

In [ ]:
def between(column, low=None, high=None):
    """Return a mask for values between low and high, inclusive."""
    values = data[column]
    mask = values.notna()
    if low is not None:
        mask = mask & (values >= low)
    if high is not None:
        mask = mask & (values <= high)
    return mask


def outside(column, low=None, high=None):
    """Return a mask for values outside a range."""
    values = data[column]
    mask = values.notna()
    if low is not None and high is not None:
        return mask & ((values < low) | (values > high))
    if low is not None:
        return mask & (values < low)
    if high is not None:
        return mask & (values > high)
    return mask


def show_mask_count(mask, name='selected galaxies'):
    print(f'{name}: {mask.sum()} galaxies out of {len(mask)}')


def describe_mask(mask, columns=properties_to_compare):
    """Show median values for a selected group of galaxies."""
    return data.loc[mask, columns].median(numeric_only=True).round(3)

In [ ]:
# Useful starter masks. Edit these or add your own.
below_ms = data['MS_CLASS'] == 'below main sequence'
on_ms = data['MS_CLASS'] == 'main sequence'
above_ms = data['MS_CLASS'] == 'above main sequence'

massive = data['LOGMSTAR'] >= 10.5
low_mass = data['LOGMSTAR'] <= 9.5
nearby = data['Z'] <= 0.1
higher_redshift = data['Z'] >= 0.3

red_galaxies = data['G_R_COLOUR'] >= data['G_R_COLOUR'].median()
blue_galaxies = data['G_R_COLOUR'] < data['G_R_COLOUR'].median()

dusty = data['AV'] >= data['AV'].quantile(0.75)
less_dusty = data['AV'] <= data['AV'].quantile(0.25)

weak_halpha = data['HALPHA_EW'] <= data['HALPHA_EW'].quantile(0.25)
strong_halpha = data['HALPHA_EW'] >= data['HALPHA_EW'].quantile(0.75)

show_mask_count(below_ms, 'below main sequence')
show_mask_count(red_galaxies & below_ms, 'red and below main sequence')
show_mask_count(dusty & above_ms, 'dusty and above main sequence')

## 6. Plot Templates

Use these functions to make lots of quick comparisons. Change the column names and masks to explore different ideas.

In [ ]:
def histogram_by_class(column, bins=40, xlim=None):
    plt.figure(figsize=(8, 5))
    for label, colour in class_colours.items():
        if label == 'not classified':
            continue
        mask = data['MS_CLASS'] == label
        sns.histplot(data.loc[mask, column], bins=bins, stat='density', element='step', fill=False,
                     linewidth=2, color=colour, label=label)
    plt.xlabel(column)
    plt.ylabel('Density')
    plt.title(f'Distribution of {column} by Main-Sequence Class')
    if xlim is not None:
        plt.xlim(*xlim)
    plt.legend()
    plt.show()


def boxplot_by_class(column):
    order = ['below main sequence', 'main sequence', 'above main sequence']
    plt.figure(figsize=(8, 5))
    sns.boxplot(data=data[data['MS_CLASS'].isin(order)], x='MS_CLASS', y=column, order=order)
    plt.xlabel('Main-sequence class')
    plt.ylabel(column)
    plt.xticks(rotation=15)
    plt.title(f'{column} by Main-Sequence Class')
    plt.show()


def scatter_by_class(x, y, xlim=None, ylim=None):
    plt.figure(figsize=(8, 6))
    for label, colour in class_colours.items():
        if label == 'not classified':
            continue
        mask = data['MS_CLASS'] == label
        plt.scatter(data.loc[mask, x], data.loc[mask, y], s=10, alpha=0.45, color=colour, label=label)
    plt.xlabel(x)
    plt.ylabel(y)
    plt.title(f'{y} vs {x}')
    if xlim is not None:
        plt.xlim(*xlim)
    if ylim is not None:
        plt.ylim(*ylim)
    plt.legend()
    plt.show()


def compare_two_masks(column, mask_a, label_a, mask_b, label_b, bins=40, xlim=None):
    plt.figure(figsize=(8, 5))
    sns.histplot(data.loc[mask_a, column], bins=bins, stat='density', element='step', fill=False, linewidth=2, label=label_a)
    sns.histplot(data.loc[mask_b, column], bins=bins, stat='density', element='step', fill=False, linewidth=2, label=label_b)
    plt.xlabel(column)
    plt.ylabel('Density')
    plt.title(f'Comparing {column}')
    if xlim is not None:
        plt.xlim(*xlim)
    plt.legend()
    plt.show()

## 7. First Trend Checks

Run these examples, then change the column names. Look for differences between below, on, and above the main sequence.

In [ ]:
histogram_by_class('G_R_COLOUR', bins=35)
boxplot_by_class('G_R_COLOUR')

In [ ]:
histogram_by_class('AV', bins=35)
boxplot_by_class('AV')

In [ ]:
histogram_by_class('HALPHA_EW', bins=40, xlim=(0, 120))
boxplot_by_class('HALPHA_EW')

In [ ]:
scatter_by_class('G_R_COLOUR', 'MS_OFFSET')
scatter_by_class('AV', 'MS_OFFSET')
scatter_by_class('HALPHA_EW', 'MS_OFFSET', xlim=(0, 150))

## 8. Build Your Own Investigation

Choose one question and use masks plus plots to investigate it.

Possible questions:

- Are below-main-sequence galaxies redder than the others?
- Are below-main-sequence galaxies more massive?
- Do above-main-sequence galaxies have stronger H-alpha equivalent widths?
- Are dusty galaxies more likely to be above or below the main sequence?
- Does the classification change with redshift?
- Do emission-line ratios differ between the classes?

In [ ]:
# Template 1: define a mask using a class.
my_mask = data['MS_CLASS'] == 'below main sequence'

show_mask_count(my_mask, 'my selected galaxies')
describe_mask(my_mask)

In [ ]:
# Template 2: define a mask using a range.
# Change the column and numbers.
my_mask = between('LOGMSTAR', low=10.0, high=11.0)

show_mask_count(my_mask, 'galaxies in my range')
describe_mask(my_mask)

In [ ]:
# Template 3: combine masks using & for AND.
# Example: galaxies that are below the main sequence AND redder than the median.
my_mask = below_ms & red_galaxies

show_mask_count(my_mask, 'below main sequence and red')
describe_mask(my_mask)

In [ ]:
# Template 4: combine masks using | for OR.
# Example: galaxies that are either above the main sequence OR have strong H-alpha emission.
my_mask = above_ms | strong_halpha

show_mask_count(my_mask, 'above main sequence or strong H-alpha')
describe_mask(my_mask)

In [ ]:
# Template 5: compare your chosen group with the rest of the sample.
my_mask = below_ms & red_galaxies
rest_mask = ~my_mask

compare_two_masks('LOGMSTAR', my_mask, 'my group', rest_mask, 'all other galaxies')
compare_two_masks('AV', my_mask, 'my group', rest_mask, 'all other galaxies')
compare_two_masks('HALPHA_EW', my_mask, 'my group', rest_mask, 'all other galaxies', xlim=(0, 150))

In [ ]:
# Template 6: make your own scatter plot.
# Try changing x_column and y_column.
x_column = 'G_R_COLOUR'
y_column = 'HALPHA_EW'

scatter_by_class(x_column, y_column, ylim=(0, 150))

## 9. Class Fractions Inside a Mask

Sometimes the most useful question is not just "what are these galaxies like?" but "which class is most common inside my selected group?"

In [ ]:
def class_fraction_table(mask, name='selected sample'):
    counts = data.loc[mask, 'MS_CLASS'].value_counts()
    fractions = counts / counts.sum()
    table = pd.DataFrame({'count': counts, 'fraction': fractions})
    print(name)
    return table

class_fraction_table(red_galaxies, 'red galaxies')

In [ ]:
# Try a few examples.
display(class_fraction_table(blue_galaxies, 'blue galaxies'))
display(class_fraction_table(dusty, 'dusty galaxies'))
display(class_fraction_table(strong_halpha, 'strong H-alpha galaxies'))
display(class_fraction_table(weak_halpha, 'weak H-alpha galaxies'))

## 10. Suggested Research Notes

Use your plots to write short notes. A good note has a claim, evidence, and a possible interpretation.

Example:

```text
Claim: Below-main-sequence galaxies appear redder.
Evidence: The G-R colour histogram is shifted to larger values for below-main-sequence galaxies.
Interpretation: Redder colours may mean older stellar populations, more dust, or less recent star formation.
```

Things to research before making strong claims:

- Why are some galaxies red and others blue?
- What is a quiescent galaxy?
- What is galaxy quenching?
- What does H-alpha equivalent width measure?
- How can dust make a star-forming galaxy look red?
- Why might more massive galaxies have different star formation histories?
- What are emission-line ratios used for?

In [ ]:
# Space for your notes:

# Claim 1:
# Evidence:
# Possible interpretation:

# Claim 2:
# Evidence:
# Possible interpretation:

# Claim 3:
# Evidence:
# Possible interpretation:

## 11. Save a Figure

Use `plt.savefig(...)` before `plt.show()` if you want to save a plot for your presentation.

In [ ]:
plt.figure(figsize=(8, 6))
for label, colour in class_colours.items():
    if label == 'not classified':
        continue
    mask = data['MS_CLASS'] == label
    plt.scatter(data.loc[mask, 'G_R_COLOUR'], data.loc[mask, 'MS_OFFSET'],
                s=10, alpha=0.45, color=colour, label=label)

plt.axhline(0, color='black', linewidth=1)
plt.axhline(0.3, color='black', linestyle='--', linewidth=1)
plt.axhline(-0.3, color='black', linestyle='--', linewidth=1)
plt.xlabel('G-R colour')
plt.ylabel('Offset from main sequence')
plt.title('Colour Compared with Main-Sequence Offset')
plt.legend()

plt.savefig('colour_vs_main_sequence_offset.png', dpi=300, bbox_inches='tight')
plt.show()